# Chapter 5: Biological Designs Evaluation

This notebook evaluates reusable biological design patterns for flow matching: multi-timepoint EB bridge sharing and condition-aware sci-Plex perturbation response prediction. Heavy data preparation for sci-Plex is intentionally cached by the Chapter 4 notebook section `sci-Plex 3 A549 data cache for Chapter 5`.


## 0. Setup

Imports, paths, environment-controlled quick/full mode, seeds, device, output directories, and save helpers.


In [1]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_ch05")

from pathlib import Path
import sys
import json
import random
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
except Exception as exc:
    raise ImportError("Chapter 5 experiments require PyTorch.") from exc

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("/home/xmabs/flow_matching_for_dynamic_biology/flow_matching_for_dynamic_biology")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import (
    load_or_prepare_sciplex3_a549,
    load_lincs_smiles_corpus,
    make_sciplex_split,
    make_sciplex_pca_state_table,
    compute_rdkit2d_with_external_norm,
)
from src.ch05_experiments import (
    set_global_seed,
    load_eb_ch05,
    run_eb_pairwise_vs_shared,
    choose_heldout_compound,
    sciplex_split_counts,
    evaluate_sciplex_split,
    aggregate_metric_table,
)

DEFAULT_SEED = int(os.environ.get("CH05_SEED", "42"))
QUICK_MODE = os.environ.get("CH05_QUICK", "1") == "1"
TRAINING_STEPS = int(os.environ.get("CH05_TRAINING_STEPS", "1500" if QUICK_MODE else "6000"))
BATCH_SIZE = int(os.environ.get("CH05_BATCH_SIZE", "128" if QUICK_MODE else "256"))
NFE = int(os.environ.get("CH05_NFE", "16" if QUICK_MODE else "32"))
EB_MAX_CELLS_PER_TIME = int(os.environ.get("CH05_EB_MAX_CELLS_PER_TIME", "220" if QUICK_MODE else "900"))
SCIPLEX_DOWNLOAD_IN_CH05 = os.environ.get("CH05_SCIPLEX_DOWNLOAD_IN_CH05", "0") == "1"
SCIPLEX_SYNTHETIC_IF_MISSING = os.environ.get("CH05_ALLOW_SYNTHETIC_SCIPLEX", "0") == "1"
MAX_EVAL_GROUPS = os.environ.get("CH05_MAX_EVAL_GROUPS", "")
MAX_EVAL_GROUPS = None if MAX_EVAL_GROUPS == "" else int(MAX_EVAL_GROUPS)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = PROJECT_ROOT / "data"
FIG_DIR = PROJECT_ROOT /  "figures" / "ch05"
TABLE_DIR = PROJECT_ROOT /  "tables" / "ch05"
OUT_DIR = PROJECT_ROOT /  "outputs" / "ch05"
for path in [FIG_DIR, TABLE_DIR, OUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def json_ready(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_ready(v) for v in obj]
    return obj

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_ready(payload), indent=2, sort_keys=True))
    return path

def save_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    return path

def save_figure(fig, filename):
    path = FIG_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path

set_global_seed(DEFAULT_SEED)
RUN_SUMMARY = {
    "quick_mode": bool(QUICK_MODE),
    "seed": int(DEFAULT_SEED),
    "device": str(DEVICE),
    "training_steps": int(TRAINING_STEPS),
    "batch_size": int(BATCH_SIZE),
    "nfe": int(NFE),
    "paths": {"figures": str(FIG_DIR), "tables": str(TABLE_DIR), "outputs": str(OUT_DIR)},
}
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}; quick={QUICK_MODE}; steps={TRAINING_STEPS}; batch={BATCH_SIZE}; nfe={NFE}")


Project root: /home/xmabs/flow_matching_for_dynamic_biology/flow_matching_for_dynamic_biology
Device: cuda; quick=False; steps=6000; batch=256; nfe=32


## 1. EB data load


In [2]:
EB_PATH = DATA_DIR / "trajectorynet_eb" / "eb_velocity_v5.npz"
eb = load_eb_ch05(EB_PATH, max_cells_per_time=EB_MAX_CELLS_PER_TIME, seed=DEFAULT_SEED, n_pc=20)
RUN_SUMMARY["eb_data"] = {
    "path": str(EB_PATH),
    "max_cells_per_time": int(EB_MAX_CELLS_PER_TIME),
    "counts": eb["counts"],
    "training_space": "standardized PC-20",
    "display_space": "PHATE 2D",
    "cluster_labels": "KMeans k=8 fit in PC-20",
}
display(eb["counts"])


,time,n_cells
0,0,900
1,1,900
2,2,900
3,3,900
4,4,900


## 2. Exp 1 EB pairwise vs shared


In [3]:
eb_metrics, eb_diag, eb_cache = run_eb_pairwise_vs_shared(
    eb,
    training_steps=TRAINING_STEPS,
    batch_size=BATCH_SIZE,
    nfe=NFE,
    seed=DEFAULT_SEED,
    device=DEVICE,
)
save_csv(TABLE_DIR / "tab_5_1_multi_timepoint.csv", eb_metrics)
save_csv(TABLE_DIR / "tab_5_1_diag_velocity_jump.csv", eb_diag)

method_order = ["pairwise_local_bridges", "shared_adjacent_only", "shared_adjacent_skip"]
method_labels = ["Pairwise", "Shared adj", "Shared adj+skip"]
method_colors = ["#4C78A8", "#54A24B", "#F58518"]
target_rows = [("hidden_t2", "hidden_t2"), ("seen_t4", "seen_t4")]
metric_cols = [("mmd_rbf", "MMD RBF"), ("sliced_w2", "Sliced W2"), ("centroid_l2", "Centroid L2")]

fig, axes = plt.subplots(2, 3, figsize=(12.5, 6.4), sharex=True)
for row_idx, (target, target_label) in enumerate(target_rows):
    target_df = eb_metrics[eb_metrics["target"].eq(target)].set_index("method").reindex(method_order)
    for col_idx, (metric, metric_label) in enumerate(metric_cols):
        ax = axes[row_idx, col_idx]
        ax.bar(method_labels, target_df[metric].to_numpy(), color=method_colors)
        ax.set_title(f"{target_label} / {metric_label}")
        ax.set_ylabel(metric_label)
        ax.tick_params(axis="x", rotation=20)
fig.suptitle("EB multi-timepoint recovery: global distribution metrics")
save_figure(fig, "fig_5_1_eb_pairwise_vs_shared.png")

RUN_SUMMARY["eb_metrics"] = eb_metrics.to_dict(orient="records")
RUN_SUMMARY["eb_diagnostics"] = eb_diag.to_dict(orient="records")
display(eb_metrics)
display(eb_diag)


,experiment,method,target,training_steps_total,steps_per_pair,mmd_rbf,sliced_w2,centroid_l2,cluster_mass_l1,rare_cluster_error,rare_clusters
0,EB multi-timepoint,pairwise_local_bridges,hidden_t2,6000,2000,0.080739,0.402712,1.115312,0.224444,0.042222,"6,3"
1,EB multi-timepoint,pairwise_local_bridges,seen_t4,6000,2000,0.007199,0.214815,0.678975,0.135556,0.000000,"3,6"
2,EB multi-timepoint,shared_adjacent_only,hidden_t2,6000,0,0.059406,0.376872,1.157762,0.477778,0.077778,"6,3"
3,EB multi-timepoint,shared_adjacent_only,seen_t4,6000,0,0.008241,0.188741,0.571825,0.042222,0.000000,"3,6"
4,EB multi-timepoint,shared_adjacent_skip,hidden_t2,6000,0,0.066241,0.383317,1.223948,0.495556,0.172222,"6,3"
5,EB multi-timepoint,shared_adjacent_skip,seen_t4,6000,0,0.007472,0.205588,0.663954,0.122222,0.000000,"3,6"


,method,boundary,velocity_jump_mean_l2,boundary_start_residual,boundary_end_residual
0,pairwise_local_bridges,t1,8.053829,1.101226,0.664030
1,shared_adjacent_only,t1,8.168670,0.000000,0.000000
2,shared_adjacent_skip,t1,5.628532,0.000000,0.000000
3,pairwise_local_bridges,t3,8.590636,1.178659,0.901966
4,shared_adjacent_only,t3,4.799015,0.000000,0.000000
5,shared_adjacent_skip,t3,2.404038,0.000000,0.000000


## 3. sci-Plex data audit + split-aware preprocessing


In [4]:
try:
    sciplex = load_or_prepare_sciplex3_a549(
        data_dir=DATA_DIR / "sciplex3_a549",
        lincs_smiles_dir=DATA_DIR / "chemcpa_lincs_smiles",
        download=SCIPLEX_DOWNLOAD_IN_CH05,
        synthetic_if_missing=SCIPLEX_SYNTHETIC_IF_MISSING,
        hvg_top_n=1000,
        seed=DEFAULT_SEED,
    )
except FileNotFoundError as exc:
    raise FileNotFoundError(
        "sci-Plex cache is missing. Run the final Chapter 4 cache section first, "
        "or set CH05_SCIPLEX_DOWNLOAD_IN_CH05=1 for this notebook."
    ) from exc

metadata = sciplex.metadata.reset_index(drop=True).copy()
source_text = str(sciplex.summary.get("source", ""))
if bool(sciplex.summary.get("is_synthetic", False)) or "synthetic" in source_text.lower():
    save_json(OUT_DIR / "real_data_audit.json", {
        "status": "failed",
        "reason": "sci-Plex cache is synthetic or synthetic-labeled",
        "summary": sciplex.summary,
    })
    raise ValueError("Chapter 5 full run refuses synthetic sci-Plex data. Rebuild the Chapter 4 cache section with real A549 data.")
heldout_compound, heldout_reason = choose_heldout_compound(metadata)
split_a = make_sciplex_split("random", metadata, test_fraction=0.2, seed=DEFAULT_SEED)
split_b = make_sciplex_split("heldout_highest_dose", metadata, seed=DEFAULT_SEED)
split_c = make_sciplex_split("heldout_compound", metadata, heldout_compound=heldout_compound, seed=DEFAULT_SEED)
splits = {
    "Split A random sanity": split_a,
    "Split B held-out highest dose": split_b,
    "Split C held-out compound": split_c,
}
split_table = sciplex_split_counts(splits)
save_csv(TABLE_DIR / "tab_5_2_sciplex_splits.csv", split_table)

states = {}
for split_name, split_meta in splits.items():
    states[split_name] = make_sciplex_pca_state_table(sciplex.adata, split_meta, n_pcs=30, hvg_top_n=1000)

cell_counts = sciplex.cell_counts.copy()
vehicle_count = int(metadata["is_vehicle"].sum())
compound_count = int(metadata.loc[~metadata["is_vehicle"], "compound"].nunique())
RUN_SUMMARY["sciplex_data"] = {
    "paths": sciplex.paths,
    "summary": sciplex.summary,
    "K_compounds": compound_count,
    "vehicle_count": vehicle_count,
    "compound_dose_counts_head": cell_counts.head(40),
    "heldout_compound": heldout_compound,
    "heldout_compound_reason": heldout_reason,
    "split_counts": split_table,
    "pca_explained_variance": {name: state.pca_explained_variance_ratio for name, state in states.items()},
}
real_data_audit = {
    "status": "ok",
    "source": sciplex.summary.get("source"),
    "source_url": sciplex.summary.get("source_url"),
    "is_synthetic": bool(sciplex.summary.get("is_synthetic", False)),
    "K_compounds": compound_count,
    "compound_list": sciplex.summary.get("compound_list", sorted(metadata.loc[~metadata["is_vehicle"], "compound"].astype(str).unique().tolist())),
    "vehicle_count": vehicle_count,
    "dose_values": sciplex.summary.get("dose_values", sorted(map(float, metadata.loc[~metadata["is_vehicle"], "dose"].dropna().unique().tolist()))),
    "missing_smiles_count": sciplex.summary.get("missing_smiles_count"),
    "obs_schema_used": sciplex.summary.get("obs_schema_used"),
    "subset_rule": sciplex.summary.get("subset_rule"),
    "split_counts": split_table,
}
save_json(OUT_DIR / "real_data_audit.json", real_data_audit)
print("K compounds:", compound_count, "vehicle cells:", vehicle_count)
print("Held-out compound:", heldout_compound, heldout_reason)
display(cell_counts.head(30))
display(split_table)


K compounds: 8 vehicle cells: 250
Held-out compound: Alisertib (MLN8237) fallback_most_cells


,compound,dose,is_vehicle,n_cells
0,DMSO,0.0,True,250
1,AG-14361,10.0,False,250
2,AG-14361,100.0,False,250
3,AG-14361,1000.0,False,209
4,AG-14361,10000.0,False,233
5,AG-490 (Tyrphostin B42),10.0,False,207
6,AG-490 (Tyrphostin B42),100.0,False,250
7,AG-490 (Tyrphostin B42),1000.0,False,205
8,AG-490 (Tyrphostin B42),10000.0,False,212
9,Alisertib (MLN8237),10.0,False,250


,split_name,split,n_cells,n_vehicle,n_treated,K_compounds,n_doses
0,Split A random sanity,test,1572,50,1522,8,4
1,Split A random sanity,train,6284,200,6084,8,4
2,Split B held-out highest dose,test,1892,0,1892,8,1
3,Split B held-out highest dose,train,5964,250,5714,8,3
4,Split C held-out compound,test,1000,0,1000,1,4
5,Split C held-out compound,train,6856,250,6606,7,4


## 4. RDKit2D preprocessing audit


In [5]:
lincs = load_lincs_smiles_corpus(cache_dir=DATA_DIR / "chemcpa_lincs_smiles", download=SCIPLEX_DOWNLOAD_IN_CH05)
compound_smiles = (
    metadata.loc[~metadata["is_vehicle"], ["compound", "SMILES"]]
    .drop_duplicates()
    .sort_values("compound")
    .reset_index(drop=True)
)
rdkit_cache_path = OUT_DIR / "rdkit2d_compound_features.npz"
rdkit_diag_path = OUT_DIR / "rdkit2d_diagnostics.json"
if rdkit_cache_path.exists() and rdkit_diag_path.exists():
    z = np.load(rdkit_cache_path, allow_pickle=True)
    cached_compounds = z["compounds"].astype(str).tolist()
    current_compounds = compound_smiles["compound"].astype(str).tolist()
    if cached_compounds == current_compounds:
        rdkit_features = z["features"].astype(np.float32)
        rdkit_diagnostics = json.loads(rdkit_diag_path.read_text())
        if int(rdkit_diagnostics.get("D_RDKit", rdkit_features.shape[1])) != int(rdkit_features.shape[1]):
            rdkit_features = None
            rdkit_diagnostics = None
    else:
        rdkit_cache_path.unlink()
        rdkit_diag_path.unlink()
        rdkit_features = None
        rdkit_diagnostics = None
else:
    rdkit_features = None
    rdkit_diagnostics = None
if rdkit_features is None:
    rdkit_result = compute_rdkit2d_with_external_norm(
        compound_smiles["SMILES"].tolist(),
        external_smiles=lincs.smiles,
    )
    rdkit_features = rdkit_result.features
    rdkit_diagnostics = rdkit_result.diagnostics
    rdkit_diagnostics["D_RDKit"] = int(rdkit_features.shape[1])
    np.savez_compressed(
        rdkit_cache_path,
        compounds=compound_smiles["compound"].astype(str).to_numpy(),
        features=rdkit_features,
    )
    save_json(rdkit_diag_path, rdkit_diagnostics)
rdkit_by_compound = {
    str(compound): rdkit_features[i]
    for i, compound in enumerate(compound_smiles["compound"].astype(str).tolist())
}
rdkit_audit = pd.DataFrame([rdkit_diagnostics])
save_csv(OUT_DIR / "rdkit2d_audit.csv", rdkit_audit)
RUN_SUMMARY["rdkit2d"] = {
    **rdkit_diagnostics,
    "D_RDKit": int(rdkit_features.shape[1]),
    "lincs_smiles_path": str(lincs.path),
    "lincs_smiles_count": len(lincs.smiles),
    "lincs_invalid_count": int(lincs.n_invalid),
}
display(rdkit_audit)


,D_RDKit,descriptor_backend,external_smiles_failure_count,n_external_smiles,n_query_smiles,nan_inf_count,smiles_failure_count,std_eps,std_too_small_count
0,200,descriptastorus,0,20329,8,75,0,1.000000e-08,4


## 5. Exp 2 Split A random sanity


In [6]:
split_a_metrics, split_a_cache = evaluate_sciplex_split(
    states["Split A random sanity"].X_pca,
    states["Split A random sanity"].metadata,
    rdkit_by_compound=rdkit_by_compound,
    split_name="Split A random sanity",
    training_steps=TRAINING_STEPS,
    batch_size=BATCH_SIZE,
    nfe=NFE,
    seed=DEFAULT_SEED,
    device=DEVICE,
    max_eval_groups=MAX_EVAL_GROUPS,
)
display(aggregate_metric_table(split_a_metrics))


,split_name,method,n_target_cells,training_steps,program_readout_mmd,program_readout_sliced_w2,program_readout_mean_abs_error,program_readout_centroid_l2,program_readout_mean_mse
0,Split A random sanity,M1_unconditional,47.5625,6000.0,0.142634,2.032241,0.595247,4.032971,0.576422
1,Split A random sanity,M2_per_compound,47.5625,6000.0,0.093434,1.419584,0.435171,2.989937,0.315287
2,Split A random sanity,M3_no_chemistry,47.5625,6000.0,0.129631,1.882723,0.568112,3.971356,0.553426
3,Split A random sanity,M4_chemistry_aware,47.5625,6000.0,0.115365,1.864676,0.574153,4.009528,0.568099
4,Split A random sanity,mean_shift,47.5625,6000.0,0.054238,0.551738,0.222201,1.504941,0.077273
5,Split A random sanity,vehicle_as_prediction,47.5625,6000.0,0.055071,0.574751,0.252550,1.720091,0.103591


## 6. Exp 3 Split B held-out highest dose


In [7]:
split_b_metrics, split_b_cache = evaluate_sciplex_split(
    states["Split B held-out highest dose"].X_pca,
    states["Split B held-out highest dose"].metadata,
    rdkit_by_compound=rdkit_by_compound,
    split_name="Split B held-out highest dose",
    training_steps=TRAINING_STEPS,
    batch_size=BATCH_SIZE,
    nfe=NFE,
    seed=DEFAULT_SEED + 1,
    device=DEVICE,
    max_eval_groups=MAX_EVAL_GROUPS,
)
display(aggregate_metric_table(split_b_metrics))


,split_name,method,n_target_cells,training_steps,program_readout_mmd,program_readout_sliced_w2,program_readout_mean_abs_error,program_readout_centroid_l2,program_readout_mean_mse
0,Split B held-out highest dose,M1_unconditional,236.5,6000.0,0.022504,0.395290,0.144356,0.978411,0.039654
1,Split B held-out highest dose,M2_per_compound,236.5,6000.0,0.024189,0.381012,0.164306,1.101221,0.046222
2,Split B held-out highest dose,M3_no_chemistry,236.5,6000.0,0.017541,0.355551,0.156063,1.053831,0.044590
3,Split B held-out highest dose,M4_chemistry_aware,236.5,6000.0,0.022031,0.384156,0.162307,1.091192,0.047263
4,Split B held-out highest dose,mean_shift,236.5,6000.0,0.025014,0.381132,0.141399,0.974032,0.038691
5,Split B held-out highest dose,vehicle_as_prediction,236.5,6000.0,0.021901,0.372205,0.165491,1.145981,0.051315


## 7. Exp 4 Split C held-out compound


In [8]:
split_c_metrics, split_c_cache = evaluate_sciplex_split(
    states["Split C held-out compound"].X_pca,
    states["Split C held-out compound"].metadata,
    rdkit_by_compound=rdkit_by_compound,
    split_name="Split C held-out compound",
    training_steps=TRAINING_STEPS,
    batch_size=BATCH_SIZE,
    nfe=NFE,
    seed=DEFAULT_SEED + 2,
    device=DEVICE,
    max_eval_groups=MAX_EVAL_GROUPS,
)
display(aggregate_metric_table(split_c_metrics))


,split_name,method,n_target_cells,training_steps,program_readout_mmd,program_readout_sliced_w2,program_readout_mean_abs_error,program_readout_centroid_l2,program_readout_mean_mse
0,Split C held-out compound,M1_unconditional,250.0,6000.0,0.098262,1.001409,0.345693,2.598519,0.237409
1,Split C held-out compound,M3_no_chemistry,250.0,6000.0,0.084126,0.965054,0.367681,2.735711,0.260844
2,Split C held-out compound,M4_chemistry_aware,250.0,6000.0,0.061913,0.802708,0.461137,3.292700,0.367572
3,Split C held-out compound,mean_shift,250.0,6000.0,0.077687,0.824166,0.340602,2.505896,0.220924
4,Split C held-out compound,nearest_chemistry,250.0,6000.0,0.086713,0.832564,0.351609,2.689945,0.251650
5,Split C held-out compound,vehicle_as_prediction,250.0,6000.0,0.063527,0.784764,0.330116,2.326546,0.188837


## 8. Merge tables, save figures


In [9]:
sciplex_metrics = pd.concat([split_a_metrics, split_b_metrics, split_c_metrics], ignore_index=True)
sciplex_summary = aggregate_metric_table(sciplex_metrics)
save_csv(OUT_DIR / "sciplex_metrics_by_group.csv", sciplex_metrics)
save_csv(OUT_DIR / "sciplex_metrics_summary.csv", sciplex_summary)

fig, ax = plt.subplots(figsize=(10.5, 4.2))
plot_df = sciplex_summary.copy()
plot_df["label"] = plot_df["split_name"].str.replace("Split ", "S", regex=False) + "\n" + plot_df["method"]
ax.bar(plot_df["label"], plot_df["program_readout_sliced_w2"], color="#4C78A8")
ax.set_ylabel("Sliced W2 in split-aware PCA-30")
ax.set_title("sci-Plex perturbation response metrics by split and method")
ax.tick_params(axis="x", rotation=75)
save_figure(fig, "fig_5_3_sciplex_heldout_compound_summary.png")

heldout_keys = [key for key in split_c_cache["predictions"] if key[0] == heldout_compound]
if not heldout_keys:
    heldout_keys = list(split_c_cache["predictions"].keys())
representative_key = sorted(heldout_keys, key=lambda x: x[1])[-1]
panel = split_c_cache["predictions"][representative_key]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), sharex=True, sharey=True)
panels = [
    ("vehicle", panel["vehicle_as_prediction"], "#8E8E8E"),
    ("ground truth", panel["target"], "#F58518"),
    ("M4 chemistry", panel["M4_chemistry_aware"], "#54A24B"),
    ("M3 no chemistry", panel["M3_no_chemistry"], "#4C78A8"),
]
for ax, (title, pts, color) in zip(axes, panels):
    pts = np.asarray(pts)
    ax.scatter(pts[:, 0], pts[:, 1], s=7, alpha=0.55, linewidths=0, color=color)
    ax.set_title(title)
    ax.set_xlabel("PC1")
axes[0].set_ylabel("PC2")
fig.suptitle(f"Split C held-out {representative_key[0]} dose={representative_key[1]:g}")
save_figure(fig, "fig_5_3_sciplex_heldout_compound.png")

RUN_SUMMARY["sciplex_metrics_summary"] = sciplex_summary
RUN_SUMMARY["sciplex_representative_heldout"] = {"compound": representative_key[0], "dose": representative_key[1]}
display(sciplex_summary)


,split_name,method,n_target_cells,training_steps,program_readout_mmd,program_readout_sliced_w2,program_readout_mean_abs_error,program_readout_centroid_l2,program_readout_mean_mse
0,Split A random sanity,M1_unconditional,47.5625,6000.0,0.142634,2.032241,0.595247,4.032971,0.576422
1,Split A random sanity,M2_per_compound,47.5625,6000.0,0.093434,1.419584,0.435171,2.989937,0.315287
2,Split A random sanity,M3_no_chemistry,47.5625,6000.0,0.129631,1.882723,0.568112,3.971356,0.553426
3,Split A random sanity,M4_chemistry_aware,47.5625,6000.0,0.115365,1.864676,0.574153,4.009528,0.568099
4,Split A random sanity,mean_shift,47.5625,6000.0,0.054238,0.551738,0.222201,1.504941,0.077273
5,Split A random sanity,vehicle_as_prediction,47.5625,6000.0,0.055071,0.574751,0.252550,1.720091,0.103591
6,Split B held-out highest dose,M1_unconditional,236.5000,6000.0,0.022504,0.395290,0.144356,0.978411,0.039654
7,Split B held-out highest dose,M2_per_compound,236.5000,6000.0,0.024189,0.381012,0.164306,1.101221,0.046222
8,Split B held-out highest dose,M3_no_chemistry,236.5000,6000.0,0.017541,0.355551,0.156063,1.053831,0.044590
9,Split B held-out highest dose,M4_chemistry_aware,236.5000,6000.0,0.022031,0.384156,0.162307,1.091192,0.047263


## 9. Write run_summary and final file existence checks


In [10]:
required_paths = [
    FIG_DIR / "fig_5_1_eb_pairwise_vs_shared.png",
    FIG_DIR / "fig_5_3_sciplex_heldout_compound.png",
    TABLE_DIR / "tab_5_1_multi_timepoint.csv",
    TABLE_DIR / "tab_5_1_diag_velocity_jump.csv",
    TABLE_DIR / "tab_5_2_sciplex_splits.csv",
    OUT_DIR / "sciplex_metrics_by_group.csv",
    OUT_DIR / "sciplex_metrics_summary.csv",
    OUT_DIR / "real_data_audit.json",
    OUT_DIR / "run_summary.json",
]

metric_frames = {
    "eb_metrics": eb_metrics,
    "eb_diag": eb_diag,
    "sciplex_metrics": sciplex_metrics,
    "sciplex_summary": sciplex_summary,
}
finite_checks = {}
for name, frame in metric_frames.items():
    numeric = frame.select_dtypes(include=[np.number])
    finite_checks[name] = bool(np.isfinite(numeric.to_numpy()).all()) if numeric.size else True

RUN_SUMMARY["key_metrics"] = {
    "eb_hidden_mmd_best": eb_metrics.loc[eb_metrics["target"].eq("hidden_t2")].sort_values("mmd_rbf").head(1).to_dict(orient="records"),
    "eb_hidden_sliced_w2_best": eb_metrics.loc[eb_metrics["target"].eq("hidden_t2")].sort_values("sliced_w2").head(1).to_dict(orient="records"),
    "eb_seen_sliced_w2_best": eb_metrics.loc[eb_metrics["target"].eq("seen_t4")].sort_values("sliced_w2").head(1).to_dict(orient="records"),
    "eb_seen_centroid_l2_best": eb_metrics.loc[eb_metrics["target"].eq("seen_t4")].sort_values("centroid_l2").head(1).to_dict(orient="records"),
    "sciplex_summary": sciplex_summary,
}
RUN_SUMMARY["finite_metric_checks"] = finite_checks
RUN_SUMMARY["expected_artifacts"] = [str(path.relative_to(PROJECT_ROOT)) for path in required_paths]
if bool(RUN_SUMMARY["sciplex_data"]["summary"].get("is_synthetic", False)):
    raise ValueError("Synthetic sci-Plex data reached final summary; refusing to write final run_summary.")
if "synthetic" in str(RUN_SUMMARY["sciplex_data"]["summary"].get("source", "")).lower():
    raise ValueError("Synthetic-labeled sci-Plex source reached final summary; refusing to write final run_summary.")
save_json(OUT_DIR / "run_summary.json", RUN_SUMMARY)

missing = []
for path in required_paths:
    if not path.exists() or path.stat().st_size <= 0:
        missing.append(str(path))
if missing:
    raise FileNotFoundError(f"Missing or empty required artifacts: {missing}")
if not all(finite_checks.values()):
    raise ValueError(f"Non-finite numeric metrics detected: {finite_checks}")

print("Required artifacts:")
for path in required_paths:
    print(path.relative_to(PROJECT_ROOT), path.stat().st_size)
display(pd.DataFrame({"metric_frame": list(finite_checks), "all_finite": list(finite_checks.values())}))


Required artifacts:
figures/ch05/fig_5_1_eb_pairwise_vs_shared.png 171896
figures/ch05/fig_5_3_sciplex_heldout_compound.png 99358
tables/ch05/tab_5_1_multi_timepoint.csv 1067
tables/ch05/tab_5_1_diag_velocity_jump.csv 449
tables/ch05/tab_5_2_sciplex_splits.csv 357
outputs/ch05/sciplex_metrics_by_group.csv 44351
outputs/ch05/sciplex_metrics_summary.csv 2932
outputs/ch05/real_data_audit.json 2262
outputs/ch05/run_summary.json 35421


,metric_frame,all_finite
0,eb_metrics,True
1,eb_diag,True
2,sciplex_metrics,True
3,sciplex_summary,True
